# Explore videos with DBP API

Connect → inspect coverage → filter → optionally download assignments.
The website must expose `/api/v1`. No presentation framework is needed.
Run cells in order. Counts come from the server's loaded dataset, not this page of results.


In [ ]:
from getpass import getpass
from pathlib import Path
import json

import pandas as pd
from IPython.display import display
from dbp_api import Client, ExperimentSpec, MetricFilter

website_url = input("Website URL [http://127.0.0.1:8773]: ").strip() or "http://127.0.0.1:8773"
client = Client(website_url, timeout=120)
client.login(input("Username: "), getpass("Password: "))
print("Connected")


## All metrics and coverage

`known` counts videos with a value, including zero or false. `unknown` means no
usable value. Each metric has its own coverage; adding these counts would double
count videos. The request returns aggregates for the full dataset, but only one
video row. No duration or content filter is applied here.


In [ ]:
inventory = client.metrics()
baseline = client.query_media(limit=1)
print(f"{baseline['matching']:,} of {baseline['total']:,} videos")

def metric_coverage(result):
    records = []
    for metric in inventory["metrics"]:
        summary = result["distributions"].get(metric["id"])
        known = summary["known"] if summary is not None else None
        records.append({
            "Metric": metric["label"],
            "ID": metric["id"],
            "Type": metric["kind"],
            "Unit": metric["unit"],
            "Operators": ", ".join(metric["operators"]),
            "Measured videos": known,
            "Unknown videos": summary["unknown"] if summary is not None else None,
            "Measured %": 100 * known / result["matching"] if known is not None and result["matching"] else None,
        })
    return pd.DataFrame(records)

with pd.option_context("display.max_rows", None):
    display(metric_coverage(baseline))


## Filter videos

Edit the content query and metric filters below. All constraints are combined.
Content search uses visual captions and spoken transcripts (`corpus="both"`);
choose `captions` or `transcripts` to restrict the source. An empty query matches
all content. Missing metric values do not satisfy numeric comparisons.


In [ ]:
content_query = ""
filters = [MetricFilter("duration_seconds", "gte", 10)]
query = dict(content_query=content_query, corpus="both", filters=filters, version=baseline["version"])
selected = client.query_media(**query, limit=20)
query["search_version"] = selected.get("search_version")
print(f"{selected['matching']:,} of {selected['total']:,} videos match; {len(selected['rows'])} shown")
display(pd.DataFrame(selected["rows"]))
with pd.option_context("display.max_rows", None):
    display(metric_coverage(selected))


## Inspect a distribution

These bins cover **all matching videos**, not just the displayed page. Baseline
and filtered counts use the same bin edges. Change the ID to any listed metric.


In [ ]:
metric_id = "duration_seconds"
distribution = selected["distributions"][metric_id]
display(pd.DataFrame(distribution["bins"]))
display(pd.DataFrame([distribution["baseline"], {
    key: distribution[key] for key in ("known", "unknown", "median", "minimum", "maximum")
}], index=["Full dataset", "Filtered"]))


## Optional: create, publish, and download

Set `download_demo = True` to save an experiment on the website and download
one subject. The server samples two parents from the filtered pool using the seed.
Publication fixes their assignment; this is not a download of every matching video.
Foils and repeats default to zero. Set `block_count` and `foils_per_block` on
`ExperimentSpec` when needed; parent count is the total across blocks.

This cell writes to the server and disk. Rerunning creates another experiment.
Choose a new download directory each time. Insufficient eligible/local media or
a changed dataset will raise an error rather than silently change the selection.


In [ ]:
download_demo = False

if download_demo:
    spec = ExperimentSpec(name="API demo", seed="demo-1", subject_count=1, parents_per_subject=2)
    experiment = client.create_experiment(spec, **query)
    publication = client.publish(experiment["id"])
    subject_id = publication["subjects"][0]["subject_id"]
    manifest_path = client.download_subject(experiment["id"], subject_id, Path("dbp-demo-download"))
    downloaded = json.loads(manifest_path.read_text())
    video_paths = [manifest_path.parent / relative for relative in downloaded["files"].values()]
    display(video_paths)
else:
    print("Download skipped. Exploration above is read-only.")


`video_paths` can be passed to your analysis or presentation package. The
manifest preserves subject, block, trial order, and foil segment assignments;
use that order for experiments. This client does not present videos or record responses.


## Disconnect
Run this when finished. Clear outputs before committing or sharing the notebook.


In [ ]:
client.logout()
print("Disconnected")
